#LSTM-based Hardware Trojan Detection Model Training

Architecture (from paper):
- Input: E-PCP traces (max_len × 100)
- LSTM Layer 1: hidden_size=128
- LSTM Layer 2: hidden_size=128  
- FC Layer: 128 → 2
- Softmax: Binary classification (HT vs Normal)

## Imports and Configs

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
!gdown 1Eht0_FNRirD7gbfydh5_SqQ1vjJ2F5Xs

Downloading...
From (original): https://drive.google.com/uc?id=1Eht0_FNRirD7gbfydh5_SqQ1vjJ2F5Xs
From (redirected): https://drive.google.com/uc?id=1Eht0_FNRirD7gbfydh5_SqQ1vjJ2F5Xs&confirm=t&uuid=10023478-bef4-443f-95ca-a95f7222f737
To: /content/e_pcp_dataset.npz
100% 152M/152M [00:00<00:00, 231MB/s]


In [2]:
import os
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from sklearn.model_selection import KFold
from sklearn.metrics import confusion_matrix
import json
from datetime import datetime
from tqdm import tqdm
import matplotlib.pyplot as plt
from pathlib import Path
import time
import warnings

## Util Functions

In [3]:
def calculate_f2_score(tp, fp, fn):
    """
    Calculate F2-score
    F2 = (1 + β²) · (precision · recall) / (β² · precision + recall)
    where β = 2 (weights recall higher than precision)
    """
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0

    beta = 2
    if (precision + recall) > 0:
        f2 = (1 + beta**2) * (precision * recall) / (beta**2 * precision + recall)
    else:
        f2 = 0.0

    return f2

In [4]:
def print_header(title, width=80):
    """Print formatted section header"""
    print("\n" + "=" * width)
    print(f"{title:^{width}}")
    print("=" * width + "\n")


def print_subheader(title, width=80):
    """Print formatted subsection header"""
    print(f"\n{title}")
    print("-" * width)


def format_time(seconds):
    """Format time in human-readable format"""
    if seconds < 60:
        return f"{seconds:.1f}s"
    elif seconds < 3600:
        mins = int(seconds // 60)
        secs = int(seconds % 60)
        return f"{mins}m {secs}s"
    else:
        hours = int(seconds // 3600)
        mins = int((seconds % 3600) // 60)
        return f"{hours}h {mins}m"


def format_memory(bytes_size):
    """Format memory size in human-readable format"""
    for unit in ['B', 'KB', 'MB', 'GB']:
        if bytes_size < 1024.0:
            return f"{bytes_size:.1f} {unit}"
        bytes_size /= 1024.0
    return f"{bytes_size:.1f} TB"

## Torch Manual Dataset

In [5]:
class EPCPDataset(Dataset):
    """PyTorch dataset for E-PCP traces"""

    def __init__(self, X, y):
        """
        Args:
            X: numpy array of shape (N, max_len, 100)
            y: numpy array of shape (N,) with labels 0/1
        """
        self.X = torch.FloatTensor(X)
        self.y = torch.LongTensor(y)

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

## Torch Model Definition

In [6]:
class LSTMHTDetector(nn.Module):
    """
    2-Layer LSTM for Hardware Trojan Detection

    Architecture from paper:
    - Input layer: Maps 100-dim to 128-dim (handled by LSTM)
    - LSTM Layer 1: input=100, hidden=128
    - LSTM Layer 2: input=128, hidden=128
    - FC layer: 128 → 2
    - Softmax: Binary classification
    """

    def __init__(
        self,
        input_size: int = 100,
        hidden_size: int = 128,
        num_layers: int = 2,
        num_classes: int = 2,
        dropout: float = 0.0
    ):
        super(LSTMHTDetector, self).__init__()

        self.hidden_size = hidden_size
        self.num_layers = num_layers

        # 2-layer LSTM
        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0
        )

        # Fully connected layer
        self.fc = nn.Linear(hidden_size, num_classes)

        # Softmax (applied during eval, loss handles it during training)
        self.softmax = nn.Softmax(dim=1)

    def forward(self, x):
        """
        Forward pass

        Args:
            x: (batch_size, seq_len, input_size)

        Returns:
            logits: (batch_size, num_classes)
        """
        # LSTM: (batch, seq, hidden)
        # We only need the last time step output
        lstm_out, (h_n, c_n) = self.lstm(x)

        # h_n shape: (num_layers, batch, hidden)
        # Take last layer's hidden state
        last_hidden = h_n[-1]  # (batch, hidden)

        # FC layer
        logits = self.fc(last_hidden)  # (batch, num_classes)

        return logits

    def predict_proba(self, x):
        """Get class probabilities"""
        logits = self.forward(x)
        return self.softmax(logits)

## Metric Tracker

In [7]:
class MetricsTracker:
    """Track and save metrics per epoch"""

    def __init__(self, output_dir: str, fold_num: int):
        self.output_dir = Path(output_dir)
        self.fold_num = fold_num
        self.metrics = {
            'train_loss': [],
            'val_loss': [],
            'val_acc': [],
            'val_tn': [],
            'val_fp': [],
            'val_fn': [],
            'val_tp': [],
            'val_tnr': [],
            'val_tpr': [],
            'val_npv': [],
            'val_ppv': [],
            'val_f2': []
        }

        # Create log file
        self.log_file = self.output_dir / f"fold{fold_num}_metrics.csv"
        self._write_header()

    def _write_header(self):
        """Write CSV header"""
        with open(self.log_file, 'w') as f:
            f.write("epoch,train_loss,val_loss,val_acc,")
            f.write("TN,FP,FN,TP,TNR,TPR,NPV,PPV,F2\n")

    def add_epoch(self, epoch, train_loss, val_loss, val_acc,
                  tn, fp, fn, tp, tnr, tpr, npv, ppv, f2):
        """Add metrics for an epoch"""
        self.metrics['train_loss'].append(train_loss)
        self.metrics['val_loss'].append(val_loss)
        self.metrics['val_acc'].append(val_acc)
        self.metrics['val_tn'].append(int(tn))
        self.metrics['val_fp'].append(int(fp))
        self.metrics['val_fn'].append(int(fn))
        self.metrics['val_tp'].append(int(tp))
        self.metrics['val_tnr'].append(float(tnr))
        self.metrics['val_tpr'].append(float(tpr))
        self.metrics['val_npv'].append(float(npv))
        self.metrics['val_ppv'].append(float(ppv))
        self.metrics['val_f2'].append(float(f2))

        # Append to CSV
        with open(self.log_file, 'a') as f:
            f.write(f"{epoch},{train_loss:.6f},{val_loss:.6f},{val_acc:.6f},")
            f.write(f"{tn},{fp},{fn},{tp},")
            f.write(f"{tnr:.6f},{tpr:.6f},{npv:.6f},{ppv:.6f},{f2:.6f}\n")

    def get_metrics(self):
        """Get all metrics"""
        return self.metrics

    def save_json(self):
        """Save metrics as JSON"""
        json_file = self.output_dir / f"fold{self.fold_num}_metrics.json"
        with open(json_file, 'w') as f:
            json.dump(self.metrics, f, indent=2)


## Model Trainer

In [8]:
class LSTMTrainer:
    """Train LSTM with K-fold cross validation and multiple balancing options"""

    def __init__(
        self,
        dataset_path: str,
        output_dir: str = "lstm_models",
        batch_size: int = 32,
        learning_rate: float = 0.001,
        num_epochs: int = 5,
        k_folds: int = 3,
        balance_method: str = 'weighted',
        device: str = 'cuda' if torch.cuda.is_available() else 'cpu'
    ):
        """
        Args:
            balance_method: One of:
                - 'upsample': Physical upsampling (2x memory, best F2~0.75)
                - 'downsample': Physical downsampling (discards data, F2~0.26)
                - 'weighted': Weighted sampling (memory-efficient, F2~0.67-0.73)
                - 'none': No balancing (baseline, F2~0.67)
        """
        self.dataset_path = dataset_path
        self.output_dir = Path(output_dir)
        self.batch_size = batch_size
        self.learning_rate = learning_rate
        self.num_epochs = num_epochs
        self.k_folds = k_folds
        self.balance_method = balance_method
        self.device = device

        # Validate balance method
        valid_methods = ['upsample', 'downsample', 'weighted', 'none']
        if balance_method not in valid_methods:
            raise ValueError(f"balance_method must be one of {valid_methods}, got '{balance_method}'")

        self.output_dir.mkdir(exist_ok=True)

        # Load dataset
        print_header("LOADING DATASET")
        print(f"Source: {dataset_path}")

        data = np.load(dataset_path, allow_pickle=True)
        self.X = data['X']
        self.y = data['y']

        ht_count = np.sum(self.y)
        normal_count = len(self.y) - ht_count
        ht_pct = np.mean(self.y) * 100

        # Calculate memory usage
        memory_mb = (self.X.nbytes + self.y.nbytes) / (1024 * 1024)

        print(f"\nDataset Statistics:")
        print(f"  Total samples:  {len(self.y):,}")
        print(f"  Feature shape:  {self.X.shape}")
        print(f"  Memory usage:   {format_memory(self.X.nbytes + self.y.nbytes)}")
        print(f"\nClass Distribution:")
        print(f"  Normal (0):     {normal_count:,} ({100-ht_pct:.2f}%)")
        print(f"  Trojan (1):     {ht_count:,} ({ht_pct:.2f}%)")
        print(f"  Imbalance ratio: {normal_count/ht_count:.1f}:1")

        print(f"\nCompute Configuration:")
        print(f"  Device:         {self.device}")
        if self.device == 'cuda':
            print(f"  GPU:            {torch.cuda.get_device_name(0)}")
            total_memory = torch.cuda.get_device_properties(0).total_memory
            print(f"  GPU Memory:     {format_memory(total_memory)}")

        # Memory warning for upsampling
        if balance_method == 'upsample':
            estimated_memory = memory_mb * 2  # Approximate 2x for upsampling
            print(f"\n  ⚠️  WARNING: Upsampling will approximately double memory usage")
            print(f"      Original: {format_memory(self.X.nbytes + self.y.nbytes)}")
            print(f"      Estimated after upsampling: ~{estimated_memory:.1f} MB")
            print(f"      Consider 'weighted' method if memory is limited")

    def upsample_minority_class(self, X_train, y_train):
        """
        Physical upsampling - duplicates minority class samples

        Pros: Best performance (Paper: F2=0.751 for combinational)
        Cons: 2x memory usage, slower training
        """
        X_normal = X_train[y_train == 0]
        X_ht = X_train[y_train == 1]

        y_normal = y_train[y_train == 0]
        y_ht = y_train[y_train == 1]

        original_ht = len(X_ht)
        original_normal = len(X_normal)
        original_total = len(y_train)
        original_ht_pct = (original_ht / original_total) * 100

        original_memory = X_train.nbytes + y_train.nbytes

        # Upsample minority class
        if len(X_ht) < len(X_normal):
            indices = np.random.choice(len(X_ht), size=len(X_normal), replace=True)
            X_ht_upsampled = X_ht[indices]
            y_ht_upsampled = y_ht[indices]

            X_balanced = np.concatenate([X_normal, X_ht_upsampled], axis=0)
            y_balanced = np.concatenate([y_normal, y_ht_upsampled], axis=0)
        else:
            X_balanced = X_train
            y_balanced = y_train

        # Shuffle
        shuffle_idx = np.random.permutation(len(X_balanced))
        X_balanced = X_balanced[shuffle_idx]
        y_balanced = y_balanced[shuffle_idx]

        balanced_ht = np.sum(y_balanced == 1)
        balanced_normal = np.sum(y_balanced == 0)
        balanced_total = len(y_balanced)
        balanced_ht_pct = (balanced_ht / balanced_total) * 100

        balanced_memory = X_balanced.nbytes + y_balanced.nbytes
        memory_increase = balanced_memory / original_memory

        print(f"\n  Class Balancing (Physical Upsampling):")
        print(f"    Before: Normal={original_normal:>7,} ({100-original_ht_pct:>5.2f}%)  " +
              f"Trojan={original_ht:>7,} ({original_ht_pct:>5.2f}%)")
        print(f"    After:  Normal={balanced_normal:>7,} ({100-balanced_ht_pct:>5.2f}%)  " +
              f"Trojan={balanced_ht:>7,} ({balanced_ht_pct:>5.2f}%)")
        print(f"    Total samples: {balanced_total:,}")
        print(f"    Memory usage:  {format_memory(balanced_memory)} ({memory_increase:.2f}x increase)")

        return X_balanced, y_balanced

    def downsample_majority_class(self, X_train, y_train):
        """
        Physical downsampling - discards majority class samples

        Pros: Memory-efficient, fast training
        Cons: Poor performance (Paper: F2=0.257, PPV=5.7%), loses data
        """
        X_normal = X_train[y_train == 0]
        X_ht = X_train[y_train == 1]

        y_normal = y_train[y_train == 0]
        y_ht = y_train[y_train == 1]

        original_ht = len(X_ht)
        original_normal = len(X_normal)
        original_total = len(y_train)
        original_ht_pct = (original_ht / original_total) * 100

        original_memory = X_train.nbytes + y_train.nbytes

        # Downsample majority class
        if len(X_normal) > len(X_ht):
            indices = np.random.choice(len(X_normal), size=len(X_ht), replace=False)
            X_normal_downsampled = X_normal[indices]
            y_normal_downsampled = y_normal[indices]

            X_balanced = np.concatenate([X_normal_downsampled, X_ht], axis=0)
            y_balanced = np.concatenate([y_normal_downsampled, y_ht], axis=0)
        else:
            X_balanced = X_train
            y_balanced = y_train

        # Shuffle
        shuffle_idx = np.random.permutation(len(X_balanced))
        X_balanced = X_balanced[shuffle_idx]
        y_balanced = y_balanced[shuffle_idx]

        balanced_ht = np.sum(y_balanced == 1)
        balanced_normal = np.sum(y_balanced == 0)
        balanced_total = len(y_balanced)
        balanced_ht_pct = (balanced_ht / balanced_total) * 100

        balanced_memory = X_balanced.nbytes + y_balanced.nbytes
        discarded_samples = original_total - balanced_total

        print(f"\n  Class Balancing (Physical Downsampling):")
        print(f"    Before: Normal={original_normal:>7,} ({100-original_ht_pct:>5.2f}%)  " +
              f"Trojan={original_ht:>7,} ({original_ht_pct:>5.2f}%)")
        print(f"    After:  Normal={balanced_normal:>7,} ({100-balanced_ht_pct:>5.2f}%)  " +
              f"Trojan={balanced_ht:>7,} ({balanced_ht_pct:>5.2f}%)")
        print(f"    Total samples: {balanced_total:,}")
        print(f"    Discarded:     {discarded_samples:,} samples ({discarded_samples/original_total*100:.1f}%)")
        print(f"    Memory usage:  {format_memory(balanced_memory)}")

        return X_balanced, y_balanced

    def create_weighted_sampler(self, y_train):
        """
        Weighted sampling - no physical data modification

        Pros: Zero memory overhead, good performance (Paper: F2~0.67-0.73)
        Cons: Slightly lower F2 than upsampling

        How it works:
        - Calculates inverse frequency weights
        - Minority class gets higher sampling probability
        - Achieves ~50-50 distribution WITHOUT duplicating data
        """
        class_counts = np.bincount(y_train)

        # Inverse frequency weights
        class_weights = 1.0 / class_counts
        sample_weights = class_weights[y_train]

        # Calculate effective distribution
        total_weight = np.sum(sample_weights)
        ht_weight_sum = np.sum(sample_weights[y_train == 1])
        effective_ht_pct = (ht_weight_sum / total_weight) * 100

        original_ht_pct = (class_counts[1] / len(y_train)) * 100

        print(f"\n  Class Balancing (Weighted Sampling):")
        print(f"    Physical data: Normal={class_counts[0]:>7,} ({100-original_ht_pct:>5.2f}%)  " +
              f"Trojan={class_counts[1]:>7,} ({original_ht_pct:>5.2f}%)")
        print(f"    Sample weights:")
        print(f"      Normal: {class_weights[0]:.6e} (lower probability)")
        print(f"      Trojan: {class_weights[1]:.6e} (higher probability)")
        print(f"    Effective distribution: ~{100-effective_ht_pct:.1f}% / {effective_ht_pct:.1f}% " +
              f"(Normal/Trojan during training)")
        print(f"    Memory overhead: 0 bytes (no duplication)")

        return WeightedRandomSampler(
            weights=sample_weights,
            num_samples=len(sample_weights),
            replacement=True
        )

    def balance_dataset(self, X_train, y_train):
        """Apply selected balancing method"""
        if self.balance_method == 'upsample':
            return self.upsample_minority_class(X_train, y_train)
        elif self.balance_method == 'downsample':
            return self.downsample_majority_class(X_train, y_train)
        elif self.balance_method in ['weighted', 'none']:
            # No physical modification for weighted sampling or no balancing
            if self.balance_method == 'none':
                print(f"\n  Class Balancing: None (baseline)")
                print(f"    Training with original imbalanced distribution")
            return X_train, y_train
        else:
            raise ValueError(f"Unknown balance method: {self.balance_method}")

    def evaluate_epoch(self, model, data_loader, criterion):
        """Evaluate model and return all metrics"""
        model.eval()

        total_loss = 0.0
        all_preds = []
        all_labels = []

        with torch.no_grad():
            for X_batch, y_batch in data_loader:
                X_batch = X_batch.to(self.device)
                y_batch = y_batch.to(self.device)

                outputs = model(X_batch)
                loss = criterion(outputs, y_batch)
                total_loss += loss.item()

                _, predicted = torch.max(outputs, 1)
                all_preds.extend(predicted.cpu().numpy())
                all_labels.extend(y_batch.cpu().numpy())

        avg_loss = total_loss / len(data_loader)

        # Confusion matrix
        cm = confusion_matrix(all_labels, all_preds)
        tn, fp, fn, tp = cm.ravel()

        # Metrics
        acc = (tn + tp) / (tn + fp + fn + tp)
        tnr = tn / (tn + fp) if (tn + fp) > 0 else 0.0
        tpr = tp / (tp + fn) if (tp + fn) > 0 else 0.0
        npv = tn / (tn + fn) if (tn + fn) > 0 else 0.0
        ppv = tp / (tp + fp) if (tp + fp) > 0 else 0.0
        f2 = calculate_f2_score(tp, fp, fn)

        return avg_loss, acc, tn, fp, fn, tp, tnr, tpr, npv, ppv, f2

    def train_fold(self, train_idx, val_idx, fold_num):
        """Train single fold"""
        print_header(f"FOLD {fold_num}/{self.k_folds}")

        fold_start_time = time.time()

        # Split data
        X_train, X_val = self.X[train_idx], self.X[val_idx]
        y_train, y_val = self.y[train_idx], self.y[val_idx]

        print("Data Split:")
        print(f"  Training:   {len(X_train):,} samples (Trojan: {np.sum(y_train):,}, {np.mean(y_train)*100:.2f}%)")
        print(f"  Validation: {len(X_val):,} samples (Trojan: {np.sum(y_val):,}, {np.mean(y_val)*100:.2f}%)")

        # Balance training set
        X_train_original = X_train  # Keep reference for memory comparison
        X_train, y_train = self.balance_dataset(X_train, y_train)

        # Create datasets
        train_dataset = EPCPDataset(X_train, y_train)
        val_dataset = EPCPDataset(X_val, y_val)

        # Create data loaders
        if self.balance_method == 'weighted':
            sampler = self.create_weighted_sampler(y_train)
            train_loader = DataLoader(
                train_dataset,
                batch_size=self.batch_size,
                sampler=sampler
            )
        else:
            train_loader = DataLoader(
                train_dataset,
                batch_size=self.batch_size,
                shuffle=True
            )

        val_loader = DataLoader(
            val_dataset,
            batch_size=self.batch_size,
            shuffle=False
        )

        # Create model
        model = LSTMHTDetector(
            input_size=100,
            hidden_size=128,
            num_layers=2,
            num_classes=2,
            dropout=0.0
        ).to(self.device)

        print(f"\nModel Configuration:")
        print(f"  Architecture: 2-Layer LSTM")
        print(f"  Input size:   100")
        print(f"  Hidden size:  128")
        print(f"  Output size:  2 (Binary)")
        print(f"  Parameters:   {sum(p.numel() for p in model.parameters()):,}")

        # Loss and optimizer
        criterion = nn.CrossEntropyLoss()
        optimizer = optim.Adam(model.parameters(), lr=self.learning_rate)

        print(f"\nTraining Configuration:")
        print(f"  Optimizer:     Adam")
        print(f"  Learning rate: {self.learning_rate}")
        print(f"  Batch size:    {self.batch_size}")
        print(f"  Epochs:        {self.num_epochs}")

        # Metrics tracker
        tracker = MetricsTracker(self.output_dir, fold_num)

        print_subheader("TRAINING PROGRESS")

        # Training loop
        for epoch in range(self.num_epochs):
            epoch_start = time.time()

            print(f"\nEpoch {epoch+1}/{self.num_epochs}")

            # Train
            model.train()
            train_loss = 0.0

            train_pbar = tqdm(
                train_loader,
                desc=f"  Training  ",
                ncols=100,
                bar_format='{l_bar}{bar}| {n_fmt}/{total_fmt} [{elapsed}<{remaining}]'
            )

            for X_batch, y_batch in train_pbar:
                X_batch = X_batch.to(self.device)
                y_batch = y_batch.to(self.device)

                optimizer.zero_grad()
                outputs = model(X_batch)
                loss = criterion(outputs, y_batch)
                loss.backward()
                optimizer.step()

                train_loss += loss.item()
                train_pbar.set_postfix({'loss': f'{loss.item():.4f}'})

            train_loss /= len(train_loader)

            # Validation
            val_loss, val_acc, tn, fp, fn, tp, tnr, tpr, npv, ppv, f2 = \
                self.evaluate_epoch(model, val_loader, criterion)

            epoch_time = time.time() - epoch_start

            # Log metrics
            tracker.add_epoch(
                epoch + 1, train_loss, val_loss, val_acc,
                tn, fp, fn, tp, tnr, tpr, npv, ppv, f2
            )

            # Print results
            print(f"\n  Results:")
            print(f"    Train Loss: {train_loss:.6f} | Val Loss: {val_loss:.6f} | Accuracy: {val_acc:.4f}")
            print(f"    Confusion Matrix: TN={tn:>6,} FP={fp:>6,} FN={fn:>6,} TP={tp:>6,}")
            print(f"    TNR: {tnr:.4f} | TPR: {tpr:.4f} | NPV: {npv:.4f} | PPV: {ppv:.4f} | F2: {f2:.4f}")
            print(f"    Epoch time: {format_time(epoch_time)}")

        fold_time = time.time() - fold_start_time

        # Save model
        model_path = self.output_dir / f"lstm_fold{fold_num}_final.pt"
        torch.save({
            'model_state_dict': model.state_dict(),
            'config': {
                'input_size': 100,
                'hidden_size': 128,
                'num_layers': 2,
                'num_classes': 2
            },
            'balance_method': self.balance_method
        }, model_path)

        # Save metrics
        tracker.save_json()

        # Plot metrics
        self.plot_fold_metrics(tracker.get_metrics(), fold_num)

        print(f"\nFold {fold_num} Summary:")
        print(f"  Total time: {format_time(fold_time)}")
        print(f"  Model saved: {model_path.name}")
        print(f"  Metrics saved: fold{fold_num}_metrics.csv, fold{fold_num}_metrics.json")
        print(f"  Plot saved: fold{fold_num}_metrics.png")

        return tracker.get_metrics()

    def plot_fold_metrics(self, metrics, fold_num):
        """Plot training curves for a fold"""
        epochs = range(1, self.num_epochs + 1)

        fig, axes = plt.subplots(2, 2, figsize=(15, 10))
        fig.suptitle(f'Fold {fold_num} Training Metrics ({self.balance_method})',
                     fontsize=16, fontweight='bold')

        # Loss
        ax = axes[0, 0]
        ax.plot(epochs, metrics['train_loss'], 'b-o', label='Train Loss', linewidth=2)
        ax.plot(epochs, metrics['val_loss'], 'r-o', label='Validation Loss', linewidth=2)
        ax.set_xlabel('Epoch', fontsize=12)
        ax.set_ylabel('Loss', fontsize=12)
        ax.set_title('Training and Validation Loss', fontsize=14, fontweight='bold')
        ax.legend(fontsize=10)
        ax.grid(True, alpha=0.3)

        # F2-Score
        ax = axes[0, 1]
        ax.plot(epochs, metrics['val_f2'], 'g-o', label='F2-Score', linewidth=2, markersize=8)
        ax.set_xlabel('Epoch', fontsize=12)
        ax.set_ylabel('F2-Score', fontsize=12)
        ax.set_title('Validation F2-Score', fontsize=14, fontweight='bold')
        ax.set_ylim([0, 1.05])
        ax.legend(fontsize=10)
        ax.grid(True, alpha=0.3)

        # TPR and TNR
        ax = axes[1, 0]
        ax.plot(epochs, metrics['val_tpr'], 'b-o', label='TPR (Recall)', linewidth=2)
        ax.plot(epochs, metrics['val_tnr'], 'r-o', label='TNR (Specificity)', linewidth=2)
        ax.set_xlabel('Epoch', fontsize=12)
        ax.set_ylabel('Rate', fontsize=12)
        ax.set_title('True Positive Rate & True Negative Rate', fontsize=14, fontweight='bold')
        ax.set_ylim([0, 1.05])
        ax.legend(fontsize=10)
        ax.grid(True, alpha=0.3)

        # PPV and NPV
        ax = axes[1, 1]
        ax.plot(epochs, metrics['val_ppv'], 'b-o', label='PPV (Precision)', linewidth=2)
        ax.plot(epochs, metrics['val_npv'], 'r-o', label='NPV', linewidth=2)
        ax.set_xlabel('Epoch', fontsize=12)
        ax.set_ylabel('Value', fontsize=12)
        ax.set_title('Positive Predictive Value & Negative Predictive Value',
                     fontsize=14, fontweight='bold')
        ax.set_ylim([0, 1.05])
        ax.legend(fontsize=10)
        ax.grid(True, alpha=0.3)

        plt.tight_layout()

        plot_path = self.output_dir / f"fold{fold_num}_metrics.png"
        plt.savefig(plot_path, dpi=300, bbox_inches='tight')
        plt.close()

    def plot_summary(self, all_fold_metrics):
        """Plot summary across all folds"""
        k = len(all_fold_metrics)
        epochs = range(1, self.num_epochs + 1)

        avg_metrics = {
            'train_loss': np.mean([m['train_loss'] for m in all_fold_metrics], axis=0),
            'val_loss': np.mean([m['val_loss'] for m in all_fold_metrics], axis=0),
            'val_f2': np.mean([m['val_f2'] for m in all_fold_metrics], axis=0),
            'val_tpr': np.mean([m['val_tpr'] for m in all_fold_metrics], axis=0),
            'val_tnr': np.mean([m['val_tnr'] for m in all_fold_metrics], axis=0),
            'val_ppv': np.mean([m['val_ppv'] for m in all_fold_metrics], axis=0),
            'val_npv': np.mean([m['val_npv'] for m in all_fold_metrics], axis=0),
        }

        fig, axes = plt.subplots(2, 2, figsize=(15, 10))
        fig.suptitle(f'Average Metrics Across {k} Folds ({self.balance_method})',
                     fontsize=16, fontweight='bold')

        # Loss
        ax = axes[0, 0]
        ax.plot(epochs, avg_metrics['train_loss'], 'b-o', label='Avg Train Loss', linewidth=2)
        ax.plot(epochs, avg_metrics['val_loss'], 'r-o', label='Avg Val Loss', linewidth=2)
        ax.set_xlabel('Epoch', fontsize=12)
        ax.set_ylabel('Loss', fontsize=12)
        ax.set_title('Average Training and Validation Loss', fontsize=14, fontweight='bold')
        ax.legend(fontsize=10)
        ax.grid(True, alpha=0.3)

        # F2
        ax = axes[0, 1]
        ax.plot(epochs, avg_metrics['val_f2'], 'g-o', label='Avg F2-Score', linewidth=2, markersize=8)
        ax.set_xlabel('Epoch', fontsize=12)
        ax.set_ylabel('F2-Score', fontsize=12)
        ax.set_title('Average Validation F2-Score', fontsize=14, fontweight='bold')
        ax.set_ylim([0, 1.05])
        ax.legend(fontsize=10)
        ax.grid(True, alpha=0.3)

        # TPR/TNR
        ax = axes[1, 0]
        ax.plot(epochs, avg_metrics['val_tpr'], 'b-o', label='Avg TPR', linewidth=2)
        ax.plot(epochs, avg_metrics['val_tnr'], 'r-o', label='Avg TNR', linewidth=2)
        ax.set_xlabel('Epoch', fontsize=12)
        ax.set_ylabel('Rate', fontsize=12)
        ax.set_title('Average TPR & TNR', fontsize=14, fontweight='bold')
        ax.set_ylim([0, 1.05])
        ax.legend(fontsize=10)
        ax.grid(True, alpha=0.3)

        # PPV/NPV
        ax = axes[1, 1]
        ax.plot(epochs, avg_metrics['val_ppv'], 'b-o', label='Avg PPV', linewidth=2)
        ax.plot(epochs, avg_metrics['val_npv'], 'r-o', label='Avg NPV', linewidth=2)
        ax.set_xlabel('Epoch', fontsize=12)
        ax.set_ylabel('Value', fontsize=12)
        ax.set_title('Average PPV & NPV', fontsize=14, fontweight='bold')
        ax.set_ylim([0, 1.05])
        ax.legend(fontsize=10)
        ax.grid(True, alpha=0.3)

        plt.tight_layout()

        plot_path = self.output_dir / "summary_metrics.png"
        plt.savefig(plot_path, dpi=300, bbox_inches='tight')
        plt.close()

    def train_k_fold(self):
        """Train with K-fold cross validation"""
        print_header("LSTM HARDWARE TROJAN DETECTION")

        training_start = time.time()

        print("Training Configuration:")
        print(f"  Cross-validation: {self.k_folds}-fold")
        print(f"  Epochs per fold:  {self.num_epochs}")
        print(f"  Batch size:       {self.batch_size}")
        print(f"  Learning rate:    {self.learning_rate}")
        print(f"  Balance method:   {self.balance_method}")
        print(f"  Output directory: {self.output_dir}")

        # Print method comparison
        print(f"\nBalance Method Comparison (from paper):")
        print(f"  'upsample':   Best F2 (0.751) but 2x memory")
        print(f"  'weighted':   Good F2 (0.67-0.73), zero memory overhead ✓")
        print(f"  'downsample': Poor F2 (0.257), loses data")
        print(f"  'none':       Baseline F2 (0.67)")

        kfold = KFold(n_splits=self.k_folds, shuffle=True, random_state=42)

        all_fold_metrics = []

        for fold, (train_idx, val_idx) in enumerate(kfold.split(self.X), 1):
            metrics = self.train_fold(train_idx, val_idx, fold)
            all_fold_metrics.append(metrics)

        # Plot summary
        self.plot_summary(all_fold_metrics)

        # Calculate average final metrics
        final_metrics = {
            'TNR': np.mean([m['val_tnr'][-1] for m in all_fold_metrics]),
            'TPR': np.mean([m['val_tpr'][-1] for m in all_fold_metrics]),
            'NPV': np.mean([m['val_npv'][-1] for m in all_fold_metrics]),
            'PPV': np.mean([m['val_ppv'][-1] for m in all_fold_metrics]),
            'F2': np.mean([m['val_f2'][-1] for m in all_fold_metrics]),
        }

        training_time = time.time() - training_start

        print_header("TRAINING COMPLETE")

        print(f"Final Performance (Epoch {self.num_epochs}, averaged across {self.k_folds} folds):\n")
        print(f"  Balance Method:            {self.balance_method}")
        print(f"  True Negative Rate (TNR):  {final_metrics['TNR']:.6f}")
        print(f"  True Positive Rate (TPR):  {final_metrics['TPR']:.6f}")
        print(f"  Negative Predictive Value: {final_metrics['NPV']:.6f}")
        print(f"  Positive Predictive Value: {final_metrics['PPV']:.6f}")
        print(f"  F2-Score:                  {final_metrics['F2']:.6f}")

        print(f"\nTraining Time: {format_time(training_time)}")

        print(f"\nOutput Files:")
        print(f"  Models:   fold1_final.pt, fold2_final.pt, fold3_final.pt")
        print(f"  Metrics:  fold1_metrics.csv/json (+ fold2, fold3)")
        print(f"  Plots:    fold1_metrics.png (+ fold2, fold3)")
        print(f"  Summary:  summary_metrics.png, training_results.json")
        print(f"  Location: {self.output_dir}/")

        # Save results
        results = {
            'final_metrics': {k: float(v) for k, v in final_metrics.items()},
            'config': {
                'k_folds': self.k_folds,
                'epochs': self.num_epochs,
                'batch_size': self.batch_size,
                'learning_rate': self.learning_rate,
                'balance_method': self.balance_method
            },
            'training_time': training_time,
            'timestamp': datetime.now().isoformat()
        }

        results_path = self.output_dir / "training_results.json"
        with open(results_path, 'w') as f:
            json.dump(results, f, indent=2)

## Driver Code

In [9]:
DATASET_PATH = "e_pcp_dataset.npz"

if not os.path.exists(DATASET_PATH):
    raise Exception("Dataset not found")

### Training with nothing

In [13]:
# Configuration
trainer = LSTMTrainer(
    dataset_path=DATASET_PATH,
    output_dir="lstm_models",
    batch_size=32,
    learning_rate=0.001,
    num_epochs=5,  # 5 epochs as requested
    k_folds=3,     # 3-fold cross validation
    device='cuda' if torch.cuda.is_available() else 'cpu'
)

Loading dataset from e_pcp_dataset.npz...
   X shape: (722531, 9, 100)
   y shape: (722531,)
   HT samples: 5902 (0.82%)
   Device: cuda



In [14]:
trainer.train_k_fold()

print(" =========== LSTM training complete! =========== ")
print("   Models saved in: lstm_models/\n  ")


LSTM TRAINING - K-FOLD CROSS VALIDATION
K-folds: 3
Epochs per fold: 5
Batch size: 32
Learning rate: 0.001

FOLD 1/3
Train: 481687 samples (HT: 3972)
Val: 240844 samples (HT: 1930)


Epoch 1/5
----------------------------------------------------------------------


Training: 100%|██████████| 15053/15053 [00:44<00:00, 339.49it/s]



Epoch 1 Results:
  Train Loss: 0.358738
  Val Loss:   0.372279
  Val Acc:    0.784354

  Confusion Matrix:
    TN: 187316  FP:  51598
    FN:    339  TP:   1591

  Metrics:
    TNR: 0.784031  TPR: 0.824352
    NPV: 0.998193  PPV: 0.029912
    F2:  0.130605

Epoch 2/5
----------------------------------------------------------------------


Training: 100%|██████████| 15053/15053 [00:43<00:00, 349.27it/s]



Epoch 2 Results:
  Train Loss: 0.280539
  Val Loss:   0.332136
  Val Acc:    0.822848

  Confusion Matrix:
    TN: 196927  FP:  41987
    FN:    679  TP:   1251

  Metrics:
    TNR: 0.824259  TPR: 0.648187
    NPV: 0.996564  PPV: 0.028933
    F2:  0.122748

Epoch 3/5
----------------------------------------------------------------------


...


Epoch 3 Results:
  Train Loss: 0.254217
  Val Loss:   0.341034
  Val Acc:    0.826086

  Confusion Matrix:
    TN: 197807  FP:  41005
    FN:    881  TP:   1150

  Metrics:
    TNR: 0.828296  TPR: 0.566224
    NPV: 0.995566  PPV: 0.027280
    F2:  0.114362

Epoch 4/5
----------------------------------------------------------------------


Training: 100%|██████████| 15053/15053 [00:43<00:00, 348.03it/s]



Epoch 4 Results:
  Train Loss: 0.244648
  Val Loss:   0.342970
  Val Acc:    0.828664

  Confusion Matrix:
    TN: 198532  FP:  40280
    FN:    985  TP:   1046

  Metrics:
    TNR: 0.831332  TPR: 0.515017
    NPV: 0.995063  PPV: 0.025311
    F2:  0.105763

Epoch 5/5
----------------------------------------------------------------------


Training: 100%|██████████| 15053/15053 [00:43<00:00, 348.46it/s]



Epoch 5 Results:
  Train Loss: 0.239480
  Val Loss:   0.318354
  Val Acc:    0.834112

  Confusion Matrix:
    TN: 199911  FP:  38901
    FN:   1052  TP:    979

  Metrics:
    TNR: 0.837106  TPR: 0.482029
    NPV: 0.994765  PPV: 0.024549
    F2:  0.101971

📊 Plot saved: fold3_metrics.png

 Summary plot saved: summary_metrics.png

K-FOLD TRAINING COMPLETE!

 Final Metrics (Epoch 5, averaged across 3 folds):
  TNR: 0.838083
  TPR: 0.496145
  NPV: 0.995071
  PPV: 0.024606
  F2:  0.102659

Results saved to: lstm_models/training_results.json

All outputs saved to: lstm_models/

 =========== LSTM training complete! =========== 
   Models saved in: lstm_models/
  


In [15]:
!cp -r lstm_models "/content/drive/MyDrive/VHDL Project Imp"

### Traning with weighted sampler

In [10]:
trainer = LSTMTrainer(
        dataset_path=DATASET_PATH,
        output_dir="lstm_models_weighted",
        batch_size=32,
        learning_rate=0.001,
        num_epochs=5,
        k_folds=3,
        balance_method='weighted',
        device='cuda' if torch.cuda.is_available() else 'cpu'
    )


                                LOADING DATASET                                 

Source: e_pcp_dataset.npz

Dataset Statistics:
  Total samples:  722,531
  Feature shape:  (722531, 9, 100)
  Memory usage:   2.4 GB

Class Distribution:
  Normal (0):     716,629 (99.18%)
  Trojan (1):     5,902 (0.82%)
  Imbalance ratio: 121.4:1

Compute Configuration:
  Device:         cuda
  GPU:            Tesla T4
  GPU Memory:     14.7 GB


In [11]:
trainer.train_k_fold()

print(" =========== LSTM training complete! =========== ")
print("   Models saved in: lstm_models/\n  ")


                         LSTM HARDWARE TROJAN DETECTION                         

Training Configuration:
  Cross-validation: 3-fold
  Epochs per fold:  5
  Batch size:       32
  Learning rate:    0.001
  Balance method:   weighted
  Output directory: lstm_models_weighted

Balance Method Comparison (from paper):
  'upsample':   Best F2 (0.751) but 2x memory
  'weighted':   Good F2 (0.67-0.73), zero memory overhead ✓
  'downsample': Poor F2 (0.257), loses data
  'none':       Baseline F2 (0.67)

                                    FOLD 1/3                                    

Data Split:
  Training:   481,687 samples (Trojan: 3,972, 0.82%)
  Validation: 240,844 samples (Trojan: 1,930, 0.80%)

  Class Balancing (Weighted Sampling):
    Physical data: Normal=477,715 (99.18%)  Trojan=  3,972 ( 0.82%)
    Sample weights:
      Normal: 2.093298e-06 (lower probability)
      Trojan: 2.517623e-04 (higher probability)
    Effective distribution: ~50.0% / 50.0% (Normal/Trojan during training)


  Training  : 100%|██████████████████████████████████████████████████████| 15053/15053 [01:16<00:00]



  Results:
    Train Loss: 0.361573 | Val Loss: 0.387706 | Accuracy: 0.7809
    Confusion Matrix: TN=186,509 FP=52,405 FN=   363 TP= 1,567
    TNR: 0.7807 | TPR: 0.8119 | NPV: 0.9981 | PPV: 0.0290 | F2: 0.1270
    Epoch time: 1m 25s

Epoch 2/5


  Training  : 100%|██████████████████████████████████████████████████████| 15053/15053 [01:11<00:00]



  Results:
    Train Loss: 0.283260 | Val Loss: 0.326550 | Accuracy: 0.8141
    Confusion Matrix: TN=194,775 FP=44,139 FN=   637 TP= 1,293
    TNR: 0.8153 | TPR: 0.6699 | NPV: 0.9967 | PPV: 0.0285 | F2: 0.1216
    Epoch time: 1m 20s

Epoch 3/5


...


  Results:
    Train Loss: 0.255035 | Val Loss: 0.342330 | Accuracy: 0.8213
    Confusion Matrix: TN=196,688 FP=42,124 FN=   921 TP= 1,110
    TNR: 0.8236 | TPR: 0.5465 | NPV: 0.9953 | PPV: 0.0257 | F2: 0.1081
    Epoch time: 1m 20s

Epoch 4/5


  Training  : 100%|██████████████████████████████████████████████████████| 15053/15053 [01:12<00:00]



  Results:
    Train Loss: 0.245528 | Val Loss: 0.324001 | Accuracy: 0.8281
    Confusion Matrix: TN=198,394 FP=40,418 FN=   976 TP= 1,055
    TNR: 0.8308 | TPR: 0.5194 | NPV: 0.9951 | PPV: 0.0254 | F2: 0.1064
    Epoch time: 1m 21s

Epoch 5/5


  Training  : 100%|██████████████████████████████████████████████████████| 15053/15053 [01:12<00:00]



  Results:
    Train Loss: 0.241175 | Val Loss: 0.343023 | Accuracy: 0.8352
    Confusion Matrix: TN=200,168 FP=38,644 FN= 1,051 TP=   980
    TNR: 0.8382 | TPR: 0.4825 | NPV: 0.9948 | PPV: 0.0247 | F2: 0.1026
    Epoch time: 1m 21s

Fold 3 Summary:
  Total time: 6m 45s
  Model saved: lstm_fold3_final.pt
  Metrics saved: fold3_metrics.csv, fold3_metrics.json
  Plot saved: fold3_metrics.png

                               TRAINING COMPLETE                                

Final Performance (Epoch 5, averaged across 3 folds):

  Balance Method:            weighted
  True Negative Rate (TNR):  0.837195
  True Positive Rate (TPR):  0.488933
  Negative Predictive Value: 0.994997
  Positive Predictive Value: 0.024135
  F2-Score:                  0.100762

Training Time: 20m 25s

Output Files:
  Models:   fold1_final.pt, fold2_final.pt, fold3_final.pt
  Metrics:  fold1_metrics.csv/json (+ fold2, fold3)
  Plots:    fold1_metrics.png (+ fold2, fold3)
  Summary:  summary_metrics.png, training_r

In [12]:
!cp -r lstm_models_weighted "/content/drive/MyDrive/VHDL Project Imp"